<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/10_safety_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10 · Safety report

Does a trained model meet our **proposed safety target**? For each hazard class and person,
at least **90% of the medium and large objects** in the test split must be found.

```
reads   MyDrive/clear_vision/exports/<run>/best.pt (+ metrics.json for the image size)
        MyDrive/clear_vision/datasets/merged.zip + merged.done   (from notebook 02)
writes  /content/datasets/merged_test/          only images/test, labels/test, data.yaml
        MyDrive/clear_vision/runs/safety_report/<model>_<date>.json
```

- **Any model:** change only `MODEL_PATH`. The image size comes from `metrics.json` next to
  it (512 for budget_v1).
- **Classes:** curb, stairs, pothole, pole, manhole, bollard, barrier, traffic_cone,
  scooter, crosswalk, and person. The list, IoU, size limits and the 90% are in
  `configs/default.yaml` → `safety_target`.
- **Found** = a prediction of the same class with IoU ≥ 0.5. Each ground-truth box can be
  found once, highest confidence first.
- **Size** = box area as a share of the image:

  | group | area | means |
  |---|---|---|
  | small | < 1% | far away: reported, not part of PASS |
  | medium | 1–5% | a few steps away |
  | large | > 5% | close |

- **Thresholds:** conf 0.15 (what the phone's Core ML NMS keeps, and the app's threshold
  for curb, stairs, pothole, manhole) and 0.25. The model runs once at 0.15. For 0.25 the
  weaker predictions are dropped, which gives the same boxes as a second run.
- **Precision** is a lower bound: each source labels only its own classes, so a correct
  detection of an unlabeled object counts as wrong.
- **Only the test files are extracted** from the zip (same helper as notebook 08).
- **Code:** `scripts/safety_report.py`, tested in `tests/test_safety_report.py`. It also runs
  on the Mac: `python scripts/safety_report.py --model models/budget_v1/best.pt --data <test data.yaml>`.
- **Runtime:** T4 GPU recommended (Runtime → Change runtime type → T4 GPU). On CPU it works
  but is slow. Run all cells.

In [ ]:
# Settings
MODEL_PATH = "/content/drive/MyDrive/clear_vision/exports/budget_v1/best.pt"  # any trained .pt
IMGSZ = None  # None = the training size from metrics.json next to the model (budget_v1: 512)
MAX_IMAGES = None  # e.g. 300 for a quick run; None = the whole test split
EXAMPLES = 8  # missed medium/large examples shown per failing class
EXAMPLE_CONF = 0.15  # misses shown at this threshold (a miss at 0.15 is also a miss at 0.25)

REPO_URL = "https://github.com/Zikirullo/clear-vision.git"
BRANCH = "develop"
DRIVE_ROOT = "/content/drive/MyDrive/clear_vision"
MERGED_ZIP = f"{DRIVE_ROOT}/datasets/merged.zip"
TEST_DEST = "/content/datasets/merged_test"  # extracted test split (Colab disk)
REPORT_DIR = f"{DRIVE_ROOT}/runs/safety_report"

## 1. Setup

In [ ]:
# 1. Setup: imports, platform, Drive, repo, packages
import datetime as dt
import logging
import os
import subprocess
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import torch

# Not Path("/kaggle").exists(): Colab runtimes have that folder too. Same rule as
# src/colab_data.py is_kaggle(); needed before the repo is cloned (checked below).
ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) and not (
    "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ)
print(f"Platform: {'Kaggle' if ON_KAGGLE else 'Colab'}")
if ON_KAGGLE:
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
else:
    from google.colab import drive

    drive.mount("/content/drive")
    if not Path(DRIVE_ROOT).exists():
        raise RuntimeError(f"Google Drive is not connected: {DRIVE_ROOT} does not exist. Re-run this cell and "
                           "allow access to Google Drive, and check that DRIVE_ROOT in Settings is right.")

REPO_DIR = Path("/kaggle/working/clear-vision" if ON_KAGGLE else "/content/clear-vision")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print(f"Repo at {REPO_DIR}: {commit.strip()}")

%pip install -q ultralytics

sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import compare_detectors as cd  # noqa: E402
import mobile_export as me  # noqa: E402
import safety_report as sr  # noqa: E402
from src.colab_data import is_kaggle  # noqa: E402
from src.detector import load_class_names  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
assert ON_KAGGLE == is_kaggle(), "platform check differs from src/colab_data.py is_kaggle()"

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU: this works but is slow. Runtime -> Change runtime type -> T4 GPU.")

MODEL_PATH = Path(MODEL_PATH)
if not MODEL_PATH.exists():
    raise RuntimeError(f"{MODEL_PATH} not found: check MODEL_PATH, or run notebook 05 (section 7 exports best.pt).")
IMGSZ = IMGSZ or me.read_imgsz(MODEL_PATH.parent / "metrics.json")
TARGET = sr.load_target(REPO_DIR / "configs" / "default.yaml")
CLASS_NAMES = load_class_names(REPO_DIR / "configs" / "classes.yaml")
print(f"Model {sr.model_label(MODEL_PATH)}: {MODEL_PATH}, imgsz {IMGSZ}")
print(f"Target: medium+large recall >= {TARGET.min_recall} at IoU {TARGET.iou}, thresholds {TARGET.conf_levels}")
print(f"Classes: {', '.join(TARGET.classes)}")

## 2. Test split

Extracts only `images/test/`, `labels/test/` and `data.yaml` from `merged.zip` (needs
`merged.done`) and reads the ground-truth boxes. A second run in the same runtime reuses
the extraction.

In [ ]:
# 2. Extract the test split from merged.zip and read the ground truth
MERGED_ZIP = Path(MERGED_ZIP)
if not (MERGED_ZIP.exists() and MERGED_ZIP.with_suffix(".done").exists()):
    raise RuntimeError(f"Need {MERGED_ZIP} and merged.done: run notebook 02 first.")

DATA_YAML = cd.extract_test_split(MERGED_ZIP, Path(TEST_DEST), CLASS_NAMES)
PAIRS = cd.load_split(DATA_YAML, "test", MAX_IMAGES)
GT = sr.load_ground_truth(PAIRS, CLASS_NAMES, Path(TEST_DEST))
counts = {c: sum(1 for boxes in GT.values() for g in boxes if g[0] == c) for c in TARGET.classes}
print(f"test images: {len(PAIRS)}")
print("ground-truth boxes: " + ", ".join(f"{c} {n}" for c, n in counts.items()))

## 3. Run the model

Once, at the lowest threshold (0.15), on every test image.

In [ ]:
# 3. Predictions on every test image
PREDS = sr.predict(MODEL_PATH, [img for img, _ in PAIRS], Path(TEST_DEST), CLASS_NAMES,
                   min(TARGET.conf_levels), IMGSZ)
print(f"{sum(len(v) for v in PREDS.values())} predictions on {len(PREDS)} images")

## 4. Recall by size, one table per threshold

`n S/M/L` = ground-truth boxes per size group. **PASS** = medium+large recall ≥ 0.90.
"no data" = no medium or large boxes of that class in the test split.

In [ ]:
# 4. One table per threshold
RESULTS = [sr.evaluate(PREDS, GT, TARGET, conf) for conf in TARGET.conf_levels]
for r in RESULTS:
    print(sr.format_table(r, TARGET), end="\n\n")

## 5. Save the results

In [ ]:
# 5. JSON on Drive: runs/safety_report/<model>_<date>.json
REPORT = sr.build_report(MODEL_PATH, IMGSZ, len(PAIRS), TARGET, RESULTS, dt.date.today().isoformat())
print(f"Saved {sr.save_report(REPORT, Path(REPORT_DIR))}")

## 6. Missed hazards

Up to 8 missed **medium or large** objects per failing class at `EXAMPLE_CONF`, from
different images where possible. **Red** = the missed ground-truth box; **yellow** = what
the model predicted in that image, so a hazard found as the wrong class shows up.

In [ ]:
# 6. Examples of missed medium/large hazards per failing class
result = next(r for r in RESULTS if r["conf"] == EXAMPLE_CONF)
failing = sr.failing_classes(result)
print(f"Failing at conf {EXAMPLE_CONF}: {', '.join(failing) or 'none'}")
for cls in failing:
    picks = sr.pick_misses(result["misses"][cls], EXAMPLES)
    cols = 4
    rows = (len(picks) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.2 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, miss in zip(axes.flat, picks):
        image = cv2.imread(str(Path(TEST_DEST) / miss["image"]))
        preds = [d for d in PREDS.get(miss["image"], []) if d.confidence >= EXAMPLE_CONF]
        ax.imshow(sr.draw_miss(image, miss, cls, preds))
        ax.set_title(f"{miss['size']} · {Path(miss['image']).name[:32]}", fontsize=8)
    r = result["classes"][cls]
    fig.suptitle(f"{cls}: medium+large recall {r['recall']['medium+large']:.2f} "
                 f"({len(result['misses'][cls])} missed)", fontsize=12)
    plt.tight_layout()
    plt.show()